__General Model Simulations__

Alfred

In [18]:
# Libraries for mathematical analysis
import numpy as np
from skimage import measure
import sympy as sym
import pylab as pl
import matplotlib.pyplot as plt
import scipy.integrate
import biocircuits

# Libraries to visualize results
import bokeh.io
from bokeh.io import show
from bokeh.plotting import figure
from bokeh.plotting import column
import bokeh.palettes
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.models import Range1d
from bokeh.io import export_svgs
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from bokeh.layouts import row

bokeh.io.output_notebook()

Loading BokehJS ...

<u>General Model</u>

![General Model](Generalized_Reaction.png)

The system is governed by the following equations:
\begin{align*}
&r^{tot} &=& r + \sum_{i=1}^{n} r_i\\
&\dot{r_1} &=& - \delta r_1 - \alpha_1 r_1\\
&\dot{r_2} &=& - \delta r_2 - \alpha_2 r_2\\
&\dot{r_3} &=& - \delta r_3 - \alpha_3 r_3\\
&\dot p &=& \alpha_3 r_3 - \phi p - apm + dc\\
&\dot m &=& \alpha_1 r_1 - \phi m - apm + dc - bzm + uq\\
&\dot z &=& \alpha_2 r_2 - \phi z - bzm + uq\\
&\dot c &=& apm  - \phi c - dc - vc\\
&\dot q &=& bzm - \phi q -  uq\\
&\dot y &=& vc - \delta y\\
\end{align*}


Where:
- $R_1 = \text{Polymerase Bound to DNA for Input RNA}$
- $R_2 = \text{Polymerase Bound to DNA for Sequestering RNA}$
- $R_3 = \text{Polymerase Bound to DNA for Target RNA}$
- $D_1 = \text{DNA Sequence for Input RNA}$
- $D_2 = \text{DNA Sequence for Sequestering RNA}$
- $D_3 = \text{DNA Sequence for Target RNA}$
- $R = \text{DNA Polymerase}$
- $M = \text{Input RNA}$
- $P = \text{Target RNA}$
- $C = \text{Input RNA-Target RNA Complex}$
- $Y = \text{Free Output RNA}$
- $Z = \text{Sequestering RNA}$
- $Q = \text{Sequestered Input RNA}$

<u>Simulated Decision Boundaries & Activation Functions<u>

In [19]:
# Function to do the 2D projection of decision boundaries with normalization  <-- From the EBCP LATAM 2025 Workshop
def contourf(p, x, y, z, title=None, palette="Spectral11", normal = True):
    """Make a filled contour plot given x, y, z data given in 2D arrays."""

    # Normalize the z values
    if normal:
      z_min = z.min()
      z_max = z.max()
      z_normalized = (z - z_min) / (z_max - z_min)  # Normalize to range [0, 1]
    else:
      z_normalized = z

    # Add zero padding at the boundaries for better visualization
    N = z_normalized.shape[1]
    z0 = np.c_[z_normalized, np.zeros(N)]
    z0[-1, -1] = 1.  # Ensure the padding contains a value within range

    # Plot the normalized values
    p.image(
        image=[z0],
        x=x.min(),
        y=y.min(),
        dw=(x.max() - x.min()) * (1 + 1 / N),
        dh=x.max() - x.min(),
        palette=palette,
        alpha=0.8,
    )

    # Color mapping based on the normalized z0 values
    color = LinearColorMapper(palette=palette, low=z0.min(), high=z0.max())
    cb = ColorBar(color_mapper=color, location=(0, 0), width=10)
    p.add_layout(cb, 'right')

    return ()

In [20]:
#Function to help calculate the ODEs
def genModelODEs(X, t, r1, r2, r3, rates):

    p, m, z, c, q, y = X

    #Unpack rates from array
    alpha1 = rates["alpha1"] # RNA production from R1
    alpha2 = rates["alpha2"] # RNA production from R2
    alpha3 = rates["alpha3"] # RNA production from R3
    phi = rates["phi"] #Decay of RNA
    delta = rates["delta"] #Dilution rate
    a = rates["a"] #Binding of input RNA & target RNA
    d = rates["d"] #Dissociation of input RNA & target RNA
    v = rates["v"] #Release of output RNA
    b = rates["b"] #Binding of input RNA & sequesterer
    u = rates["u"] #Dissociation of input RNA & sequesterer

    #By mass conservation
    #r = rtot - r3 - r2 - r1

    #Write out the ODEs
    dr1 = - delta*r1 - alpha1*r1
    dr2 = - delta*r2 - alpha2*r2
    dr3 = - delta*r3 - alpha3*r3
    dp = alpha3*r3 - phi*p - a*p*m + d*c
    dm = alpha1*r1 - phi*m - a*p*m + d*c - b*z*m + u*q
    dz = alpha2*r2 - phi*z - b*z*m + u*q
    dc = a*p*m - phi*c - d*c - v*c
    dq = b*z*m - phi*q - u*q
    dy = v*c - delta*y

    return [dp, dm, dz, dc, dq, dy]

In [21]:
# Define a grid of input values (d1, d2)
r1 = np.linspace(0, 1, 50)
r2 = np.linspace(0, 1, 50)
R1, R2 = np.meshgrid(r1, r2)
R3 = 1

#Activation function inputs
r1Vals = np.linspace(0, 1, 50)
r2Vals = [5, 3, 1]

# Simulation time
t = np.linspace(0, 200, 400)

#Initial species states
X0 = [0, #p
      0, #m
      0, #z
      0, #c
      0, #q
      0] #y

#Kinetics parameters
ratesDefault = {
    'alpha3': np.array([0.9, 1, 1.1]),
    'alpha1': np.array([0.9, 1, 1.1]),
    'alpha2': np.array([0.9, 1, 1.1]),
    'phi': np.array([4.5, 5, 5.5]),
    'delta': np.array([0.9, 1, 1.1]),
    'a': np.array([90, 100, 110]),
    'd': np.array([0.9, 1, 1.1]),
    'v': np.array([9, 10, 11]),
    'b': np.array([900, 1000, 1100]),
    'u': np.array([0.9, 1, 1.1])
}

In [22]:
#getVariantRates() --> Extracts variant-specific rates from ratesDefault
def getVariantRates(ratesArray, variantIndex):
    rates = {}
    for key, arr in ratesArray.items():
        rates[key] = arr[variantIndex]
    return rates

numVariants = len(ratesDefault['phi']) #phi is in all models --> Best choice

#Bokeh color palette for heatmaps + activation functions
colors = bokeh.palettes.OrRd3

#Array to store all results
results = []

#Range through each variant --> Solve ODEs + Plot heatmap + Plot activation function --> Append to Results
for variantIndex in range(numVariants):
    #Get rates for this current iteration's variant
    rates = getVariantRates(ratesDefault, variantIndex)

    #ODE SOLUTIONS
    #Array for storing scipy.integrate solutions
    gridSolutions = [[None for _ in range(len(r1))] for _ in range(len(r2))]
    
    # Initialize a matrix to store the steady-state output y
    Y = np.zeros_like(R1)

    # Iterate over the grid values for all variable input D1 and D2 --> Solve the ODEs
    for i in range(len(r1)):
        for j in range(len(r2)):
            solutions = scipy.integrate.odeint(genModelODEs, X0, t, args=(R1[j, i], R2[j, i], R3, rates))
            gridSolutions[j][i] = solutions
            Y[j, i] = solutions[-1, 5] #Extract the steady state output y

    #HEATMAPS
    p_heatmap = figure(width=350, height=300, title=f"Variant {variantIndex}")

    #Make the heatmap
    contourf(
        p_heatmap,
        R1,
        R2,
        Y,
        palette=bokeh.palettes.Oranges8[::-1],
        normal=True
    )

    p_heatmap.xaxis.axis_label = "R1"
    p_heatmap.yaxis.axis_label = "R2"

    #ACTIVATION FUNCTIONS
    activationFunctions = {}
    p_activation = figure(width=350, height=300, title=f"Variant {variantIndex}")

    for r2j, color in zip(r2Vals, colors):

        curves = []
        output_y = np.zeros_like(r1Vals)

        for i, r1i in enumerate(r1Vals):

            sol = scipy.integrate.odeint(
                genModelODEs, X0, t,
                args=(r1i, r2j, R3, rates)
            )

            curves.append(sol)
            output_y[i] = sol[-1, 5]

        activationFunctions[r2j] = curves

        p_activation.line(
            r1Vals,
            output_y,
            line_width=3,
            color=color,
            legend_label=f"D2 = {r2j}"
        )

    p_activation.xaxis.axis_label = "R1"
    p_activation.yaxis.axis_label = "y"
    p_activation.legend.location = "top_left"

    #Append ODE solutions, heatmaps, and activation functions for all variants into the results array
    results.append({
        "variantIndex": variantIndex,
        "rates": rates,
        "solutions": {
            "grid": gridSolutions,
            "activation": activationFunctions
        },
        "heatmaps": p_heatmap,
        "activationFunctions": p_activation
    })

In [23]:
#Plot results
for res in results:
    print(f"\nVariant {res['variantIndex']}")
    for k, v in res["rates"].items():
        print(f"{k}: {v}")

    show(row(res["heatmaps"], res["activationFunctions"]))


Variant 0
alpha3: 0.9
alpha1: 0.9
alpha2: 0.9
phi: 4.5
delta: 0.9
a: 90
d: 0.9
v: 9
b: 900
u: 0.9



Variant 1
alpha3: 1.0
alpha1: 1.0
alpha2: 1.0
phi: 5.0
delta: 1.0
a: 100
d: 1.0
v: 10
b: 1000
u: 1.0



Variant 2
alpha3: 1.1
alpha1: 1.1
alpha2: 1.1
phi: 5.5
delta: 1.1
a: 110
d: 1.1
v: 11
b: 1100
u: 1.1
